# Spectral Angle Mapper (SAM) and Sentinel-2 Calibration

This notebook implements the Spectral Angle Mapper (SAM) algorithm for hyperspectral/multispectral classification and demonstrates how Sentinel-2 data can be calibrated using high-resolution airborne hyperspectral data.

## 1. Setup and Data Loading

We start by importing necessary libraries and defining paths to our data sources.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from glob import glob
import spectral.io.envi as envi
from scipy.stats import linregress

# Set paths
AIRBORNE_DATA_DIR = '../data/images/'
SPECTRA_PATH = 'spectral_signatures/'

# Sentinel-2 central wavelengths (approximate nm)
S2_WAVELENGTHS = {
    'B02': 490, # Blue
    'B03': 560, # Green
    'B04': 665, # Red
    'B05': 705, # RE1
    'B06': 740, # RE2
    'B07': 783, # RE3
    'B08': 842, # NIR
    'B8A': 865, # NIR Narrow
    'B11': 1610, # SWIR1
    'B12': 2190  # SWIR2
}

## 2. Spectral Angle Mapper (SAM) Implementation

SAM is a physically-based spectral classification that uses an n-D angle to match pixels to reference spectra. The algorithm determines the spectral similarity between two spectra by calculating the angle between the spectra, treating them as vectors in a space with dimensionality equal to the number of bands.

The spectral angle $\alpha$ is calculated as:
$$\alpha = \cos^{-1} \left( \frac{\sum_{i=1}^{n} t_i r_i}{\sqrt{\sum_{i=1}^{n} t_i^2} \sqrt{\sum_{i=1}^{n} r_i^2}} \right)$$
where $t$ is the target spectrum (pixel) and $r$ is the reference spectrum.

In [ ]:
def compute_sam(image, ref_spectrum):
    """
    Compute Spectral Angle Mapper (SAM) between an image and a reference spectrum.
    
    Parameters:
    image: numpy array of shape (rows, cols, bands)
    ref_spectrum: numpy array of shape (bands,)
    
    Returns:
    angle_map: numpy array of shape (rows, cols) containing the spectral angle in radians
    """
    rows, cols, bands = image.shape
    # Flatten the image to (pixels, bands)
    pixels = image.reshape(-1, bands)
    
    # Calculate dot product: (pixels, bands) * (bands,) -> (pixels, bands) -> sum -> (pixels,)
    dot_product = np.sum(pixels * ref_spectrum, axis=1)
    
    # Calculate norms
    norm_pixels = np.sqrt(np.sum(pixels**2, axis=1))
    norm_ref = np.sqrt(np.sum(ref_spectrum**2))
    
    # Calculate cosine of the angle
    cos_theta = dot_product / (norm_pixels * norm_ref + 1e-8)
    
    # Clip to avoid numerical errors
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    
    # Calculate angle
    angle = np.arccos(cos_theta)
    
    return angle.reshape(rows, cols)

## 3. Loading Reference Spectra and Resampling

We load the reference spectral signatures from CSV files. These signatures are used as the reference vectors for the SAM algorithm.

In [ ]:
def load_reference_spectra(path):
    sig_files = glob(os.path.join(path, '*.csv'))
    spectra = {}
    for f in sig_files:
        # Extract class name from filename (e.g., 'forest1.csv' -> 'forest')
        class_name = os.path.basename(f).replace('.csv', '').rstrip('1234567890')
        df = pd.read_csv(f)
        if class_name not in spectra: 
            spectra[class_name] = []
        spectra[class_name].append(df['value'].values)
    
    # Average spectra for each class
    reference_spectra = {k: np.nanmean(v, axis=0) for k, v in spectra.items()}
    return reference_spectra

def resample_spectrum(spectrum, original_wavelengths, target_wavelengths):
    """
    Simple linear interpolation to resample a spectrum from hyperspectral bands to multispectral bands.
    """
    return np.interp(target_wavelengths, original_wavelengths, spectrum)

## 4. Sentinel-2 Calibration using Airborne Data

Calibration involves finding the relationship between Sentinel-2 reflectance and the higher-quality airborne data. Airborne sensors usually have much higher spectral resolution (hundreds of narrow bands) compared to Sentinel-2 (13 broader bands). By matching Sentinel-2 bands to the corresponding airborne bands, we can determine a linear correction (slope and offset) to apply to the satellite data.

This helps in reducing atmospheric effects and sensor-specific biases, making satellite data more consistent with high-accuracy field or airborne measurements.

In [ ]:
def calibrate_bands(s2_data, airborne_data, s2_band_names, airborne_wavelengths):
    """
    Calibrates Sentinel-2 bands using airborne data.
    
    Parameters:
    s2_data: dict of {band_name: 2D array}
    airborne_data: 3D hypercube (rows, cols, bands)
    s2_band_names: list of S2 band names to calibrate
    airborne_wavelengths: wavelengths of the airborne bands
    """
    calibrated_s2 = {}
    models = {}
    
    for band_name in s2_band_names:
        target_wl = S2_WAVELENGTHS[band_name]
        # Find nearest airborne band
        ab_band_idx = np.argmin(np.abs(airborne_wavelengths - target_wl))
        
        s2_pixels = s2_data[band_name].flatten()
        ab_pixels = airborne_data[:, :, ab_band_idx].flatten()
        
        # Remove NaNs and outliers for regression
        mask = (~np.isnan(s2_pixels)) & (~np.isnan(ab_pixels))
        x, y = s2_pixels[mask], ab_pixels[mask]
        
        # Linear regression: airborne = slope * s2 + intercept
        # We assume airborne data is the 'truth' or reference.
        slope, intercept, r_value, p_value, std_err = linregress(x, y)
        
        models[band_name] = {'slope': slope, 'intercept': intercept, 'r2': r_value**2}
        calibrated_s2[band_name] = slope * s2_data[band_name] + intercept
        
        print(f"Calibrated {band_name} using airborne band {ab_band_idx} ({airborne_wavelengths[ab_band_idx]:.1f}nm)")
        print(f"  R² = {r_value**2:.4f}, slope = {slope:.4f}, intercept = {intercept:.4f}")
        
    return calibrated_s2, models

## 5. Integrated Classification Workflow

The full workflow combines calibration and SAM classification:
1. Load airborne data and its spectral metadata.
2. Load overlapping Sentinel-2 data.
3. Calibrate Sentinel-2 bands to match the airborne reflectance scale.
4. Resample hyperspectral reference signatures to the multispectral Sentinel-2 band configuration.
5. Use SAM to classify the calibrated Sentinel-2 imagery.

In [ ]:
def run_integrated_workflow():
    print("Starting integrated SAM & Calibration workflow...")
    
    # 1. Search for airborne data
    hdrs = glob(os.path.join(AIRBORNE_DATA_DIR, '*.hdr'))
    if not hdrs:
        print("No airborne data found. Please ensure data is in 'data/images/'.")
        return
    
    # 2. Load airborne metadata and signatures
    img = envi.open(hdrs[0])
    airborne_wavelengths = np.array([float(x) for x in img.metadata['wavelength']])
    ref_spectra_hyper = load_reference_spectra(SPECTRA_PATH)
    
    # 3. Simulate/Load Sentinel-2 data (Placeholder)
    # In a real case, you would load these from TIF files
    print("Note: In a complete environment, we would load real Sentinel-2 patches here.")
    
    # 4. Demonstrate resampling of reference spectra to Sentinel-2
    s2_bands_to_use = ['B02', 'B03', 'B04', 'B08']
    s2_target_wavelengths = [S2_WAVELENGTHS[b] for b in s2_bands_to_use]
    
    ref_spectra_s2 = {}
    for name, spec in ref_spectra_hyper.items():
        ref_spectra_s2[name] = resample_spectrum(spec, airborne_wavelengths, s2_target_wavelengths)
        print(f"Resampled {name} signature to Sentinel-2 bands.")
    
    print("\nReady for classification on calibrated data.")

run_integrated_workflow()